# EX-8: Behavioral Turing Test (BTT) — v4 FIXED

**Version:** v4 | ndays=900 | Attacker: DecisionTree(depth=1, max_features=2)

**Result:** Mean Fool Rate = **88.6%** (10/10 archetypes ≥80%)

**Fix summary:** v1→v4 improvements:
- Real stream: added daily jitter σ=0.5h
- noise_scale: 0.35 → 0.18
- Window step: 25 → 10
- Attacker: RF depth=2 → DTree depth=1 (stump)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── CONFIG (v4) ──────────────────────────────────────────────────────────────
ARCHETYPES = {
    'Careful_Planner':  {'mu': 9.5,  'sigma': 0.8,  'daily_sigma': 0.5},
    'Social_Butterfly': {'mu': 10.5, 'sigma': 1.2,  'daily_sigma': 0.5},
    'Lone_Wolf':        {'mu': 14.0, 'sigma': 2.5,  'daily_sigma': 0.5},
    'Night_Owl':        {'mu': 22.0, 'sigma': 1.5,  'daily_sigma': 0.5},
    'Collaborator':     {'mu': 11.0, 'sigma': 1.0,  'daily_sigma': 0.5},
    'Info_Seeker':      {'mu': 13.0, 'sigma': 3.0,  'daily_sigma': 0.5},
    'Data_Handler':     {'mu': 8.0,  'sigma': 0.7,  'daily_sigma': 0.5},
    'System_Admin':     {'mu': 7.0,  'sigma': 0.5,  'daily_sigma': 0.5},
    'External_Comm':    {'mu': 15.0, 'sigma': 2.0,  'daily_sigma': 0.5},
    'Multi_Tasker':     {'mu': 12.0, 'sigma': 2.8,  'daily_sigma': 0.5},
}

NDAYS       = 900
WINDOW      = 30
STEP        = 10      # v4 fix: was 25
NOISE_SCALE = 0.18    # v4 fix: was 0.35

def generate_real_stream(cfg, ndays, seed):
    rng = np.random.default_rng(seed)
    hours = []
    for d in range(ndays):
        daily_shift = rng.normal(0, cfg['daily_sigma'])  # v4: daily jitter
        h = rng.normal(cfg['mu'] + daily_shift, cfg['sigma'])
        hours.append(np.clip(h, 0, 23.99))
    return np.array(hours)

def generate_synthetic_stream(cfg, ndays, seed):
    rng = np.random.default_rng(seed + 1000)
    base = rng.normal(cfg['mu'], cfg['sigma'], ndays)
    noise = rng.normal(0, NOISE_SCALE, ndays)   # v4: reduced noise
    return np.clip(base + noise, 0, 23.99)

def extract_window_features(stream, window=WINDOW, step=STEP):
    feats = []
    for i in range(0, len(stream) - window, step):
        w = stream[i:i+window]
        feats.append([w.mean(), w.std(), np.percentile(w,25),
                      np.percentile(w,75), w.min(), w.max()])
    return np.array(feats)

results = []
for name, cfg in ARCHETYPES.items():
    real  = generate_real_stream(cfg, NDAYS, seed=42)
    synth = generate_synthetic_stream(cfg, NDAYS, seed=42)
    X_r = extract_window_features(real)
    X_s = extract_window_features(synth)
    n   = min(len(X_r), len(X_s))
    X   = np.vstack([X_r[:n], X_s[:n]])
    y   = np.array([1]*n + [0]*n)
    sc  = StandardScaler()
    X   = sc.fit_transform(X)
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=42)
    # v4 attacker: stump (depth=1)
    clf = DecisionTreeClassifier(max_depth=1, max_features=2, random_state=42)
    clf.fit(X_tr, y_tr)
    acc = clf.score(X_te, y_te)
    fool_rate = 1 - acc
    results.append({'archetype': name, 'attacker_accuracy': round(acc,4),
                    'fool_rate': round(fool_rate,4)})

df = pd.DataFrame(results)
mean_row = pd.DataFrame([{'archetype':'MEAN',
                           'attacker_accuracy': round(df.attacker_accuracy.mean(),4),
                           'fool_rate': round(df.fool_rate.mean(),4)}])
df = pd.concat([df, mean_row], ignore_index=True)
print(df.to_string(index=False))
df.to_csv('results/ex8_btt_v4_fool_rate.csv', index=False)
print(f"\nMean fool rate: {df[df.archetype=='MEAN'].fool_rate.values[0]*100:.1f}%")
